# 🎸 Automatic Music Instrument Identification
### CNN + BiLSTM · Multi-label Classification · IRMAS Dataset

---

**Pipeline overview:**
```
.wav file
  └─► Mel Spectrogram [128 × 130]
        └─► CNN (local frequency patterns)
              └─► BiLSTM (temporal dynamics)
                    └─► Linear → Sigmoid → 11 probabilities
                          └─► threshold → [cel, flu, pia, ...]
```

| Stage | What happens |
|---|---|
| 0 | Dataset exploration |
| 1 | Raw audio waveform |
| 2 | Mel spectrogram extraction |
| 3 | Dataset & DataLoader tensors |
| 4 | Model architecture |
| 5 | Training loop |
| 6 | Inference on samples |
| 7 | Evaluation metrics |

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm

# make src/ importable
ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "src"))

from dataset import (
    CLASSES, CLASS_TO_IDX, NUM_CLASSES,
    load_training_dataset, train_val_split, IRMASDataset,
)
from features import compute_mel_spectrogram, SAMPLE_RATE, N_MELS, N_FFT, HOP_LENGTH, TIME_FRAMES
from model import InstrumentCNNBiLSTM

TRAIN_ROOT = ROOT / "data" / "IRMAS" / "training"
CHECKPOINT = ROOT / "model.pt"
THRESHOLD  = 0.09

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# dark theme for all plots
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#3a3d4d",
    "axes.labelcolor":  "#c0c4d8",
    "xtick.color":      "#7a7e94",
    "ytick.color":      "#7a7e94",
    "text.color":       "#c0c4d8",
    "grid.color":       "#2a2d3d",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "legend.facecolor": "#1a1d27",
    "legend.edgecolor": "#3a3d4d",
})

ACCENT  = "#4fc3f7"
GREEN   = "#81c784"
ORANGE  = "#ff8a65"
RED     = "#ef5350"
MUTED   = "#546e7a"

CLASS_LABELS = {
    "cel": "Cello",       "cla": "Clarinet",   "flu": "Flute",
    "gac": "Ac. Guitar",  "gel": "El. Guitar",  "org": "Organ",
    "pia": "Piano",       "sax": "Saxophone",   "tru": "Trumpet",
    "vio": "Violin",      "voi": "Voice",
}

def find_root(base):
    if any((base / c).is_dir() for c in CLASSES):
        return base
    for child in sorted(base.iterdir()):
        if child.is_dir() and any((child / c).is_dir() for c in CLASSES):
            return child
    raise RuntimeError(f"No instrument folders under {base}")

train_root = find_root(TRAIN_ROOT)
print(f"Device  : {device}")
print(f"Root    : {train_root}")
print(f"Classes : {CLASSES}")

---
## Stage 0 — Dataset Distribution

In [ ]:
counts  = {cls: len(list((train_root / cls).glob("*.wav"))) for cls in CLASSES}
all_wavs = {cls: sorted((train_root / cls).glob("*.wav")) for cls in CLASSES}

fig, ax = plt.subplots(figsize=(11, 4))
fig.suptitle("Stage 0 — Dataset Distribution", fontsize=14, fontweight="bold", color="white")

labels = [CLASS_LABELS[c] for c in CLASSES]
values = [counts[c] for c in CLASSES]
colors = plt.cm.cool(np.linspace(0.2, 0.8, len(CLASSES)))

bars = ax.barh(labels, values, color=colors, edgecolor="#0f1117", linewidth=0.8)
for bar, v in zip(bars, values):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2,
            str(v), va="center", fontsize=9, color="#c0c4d8")

ax.set_xlabel("Number of .wav files")
ax.set_xlim(0, max(values) * 1.12)
ax.invert_yaxis()
ax.grid(axis="x")
ax.set_title(f"Total: {sum(values)} files · {NUM_CLASSES} classes · ~{sum(values)//NUM_CLASSES} per class",
             fontsize=9, color=MUTED)
plt.tight_layout()
plt.show()

print(f"{'Class':<10} {'Files':>6}")
print("-" * 18)
for cls in CLASSES:
    print(f"{cls:<10} {counts[cls]:>6}")

---
## Stage 1 — Raw Audio Waveform

Аудіо — це масив амплітуд у часі. Кожен файл IRMAS: **~3 секунди** при sr=44100 Hz → ~132,000 відліків.

In [ ]:
# pick one file per instrument for later use
sample_wavs = {cls: all_wavs[cls][0] for cls in CLASSES}
demo_wav    = sample_wavs["gel"]   # electric guitar — visually interesting

y_raw, sr = sf.read(demo_wav, always_2d=True)
y_mono    = y_raw.mean(axis=1)
t         = np.linspace(0, len(y_mono) / sr, len(y_mono))

print(f"File       : {demo_wav.name}")
print(f"Label      : {demo_wav.parent.name}")
print(f"Sample rate: {sr} Hz")
print(f"Duration   : {len(y_mono)/sr:.2f} s")
print(f"Samples    : {len(y_mono):,}")
print(f"Amplitude  : [{y_mono.min():.3f}, {y_mono.max():.3f}]")

fig, axes = plt.subplots(2, 1, figsize=(13, 5),
                          gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle(f"Stage 1 — Raw Audio Waveform · {demo_wav.name}  (label: {demo_wav.parent.name})",
             fontsize=12, fontweight="bold", color="white")

ax = axes[0]
ax.plot(t, y_mono, color=ACCENT, linewidth=0.35, alpha=0.85)
ax.fill_between(t, y_mono, alpha=0.12, color=ACCENT)
ax.axhline(0, color=MUTED, linewidth=0.5)
ax.set_ylabel("Amplitude")
ax.set_xlim(0, t[-1])
ax.set_ylim(-1.05, 1.05)
ax.grid()

frame = sr // 100
rms   = np.array([np.sqrt(np.mean(y_mono[i:i+frame]**2))
                  for i in range(0, len(y_mono) - frame, frame)])
t_rms = np.linspace(0, len(y_mono)/sr, len(rms))
ax2   = axes[1]
ax2.fill_between(t_rms, rms, color=ORANGE, alpha=0.75)
ax2.set_ylabel("RMS energy")
ax2.set_xlabel("Time (s)")
ax2.set_xlim(0, t[-1])
ax2.grid()

plt.tight_layout()
plt.show()

### Stage 1 — як виглядає аудіо математично

Аудіо — це звичайний **одновимірний масив float32**. Кожне число = тиск повітря в конкретний момент часу, нормалізований до діапазону `[-1.0, +1.0]`.

```
y = [y₀, y₁, y₂, ..., y_{T-1}]   shape: (132299,)
         ↑
     44100 таких значень на секунду
```

In [ ]:
N_SHOW = 40   # скільки семплів показати

print(f"Тип масиву : {y_mono.dtype}")
print(f"Розмір     : {y_mono.shape}  ({len(y_mono):,} відліків)")
print(f"Діапазон   : [{y_mono.min():.4f}, {y_mono.max():.4f}]")
print()
print(f"Перші {N_SHOW} відліків (індекси 0–{N_SHOW-1}):")
print()

# друкуємо як рядки по 8
for row_start in range(0, N_SHOW, 8):
    row = y_mono[row_start:row_start+8]
    idx_str  = f"  [{row_start:>5}]"
    vals_str = "  ".join(f"{v:+.5f}" for v in row)
    print(f"{idx_str}  {vals_str}")

print()
print(f"  ...  ({len(y_mono) - N_SHOW:,} відліків пропущено)")
print()
print(f"Останні 8 відліків:")
last = y_mono[-8:]
print("  [" + "  ".join(f"{v:+.5f}" for v in last) + "]")

# zoom-in: показуємо перші N_SHOW семплів з позначеними значеннями
fig, ax = plt.subplots(figsize=(14, 4))
fig.suptitle(f"Stage 1 (matrix view) — перші {N_SHOW} відліків з числовими значеннями",
             fontsize=12, fontweight="bold", color="white")

xs = np.arange(N_SHOW)
ys = y_mono[:N_SHOW]

ax.stem(xs, ys, linefmt=ACCENT, markerfmt="o", basefmt=MUTED)
ax.axhline(0, color=MUTED, lw=0.8)

for i, v in enumerate(ys):
    va = "bottom" if v >= 0 else "top"
    offset = 0.02 if v >= 0 else -0.02
    ax.text(i, v + offset, f"{v:.3f}", ha="center", fontsize=6.5,
            color="#c0c4d8", rotation=70)

ax.set_xlabel("Індекс відліку (sample index)")
ax.set_ylabel("Амплітуда")
ax.set_xlim(-0.5, N_SHOW - 0.5)
ax.set_ylim(-1.15, 1.15)
ax.grid(axis="y")
ax.set_title(f"1 відлік = 1/44100 сек ≈ 22.7 мкс", fontsize=9, color=MUTED)
plt.tight_layout()
plt.show()

### Порівняння waveforms по класах

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 7))
fig.suptitle("Stage 1b — Waveforms by Instrument Class",
             fontsize=13, fontweight="bold", color="white")

colors_cls = plt.cm.cool(np.linspace(0.15, 0.85, NUM_CLASSES))
all_axes   = axes.flatten()

for idx, cls in enumerate(CLASSES):
    wav_path = sample_wavs[cls]
    y, sr_   = sf.read(wav_path, always_2d=True)
    y        = y.mean(axis=1)
    t_       = np.linspace(0, len(y)/sr_, len(y))
    ax = all_axes[idx]
    ax.plot(t_, y, color=colors_cls[idx], linewidth=0.3, alpha=0.9)
    ax.fill_between(t_, y, alpha=0.1, color=colors_cls[idx])
    ax.set_title(f"{CLASS_LABELS[cls]}  ({cls})", fontsize=9, color="white")
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks([])
    ax.set_yticks([])

all_axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

---
## Stage 2 — Mel Spectrogram

Перетворення аудіо у 2D-зображення:

```
wav [T samples]
  └─► STFT (n_fft=1024, hop=512)   → частотно-часове представлення
        └─► Mel filterbank (128 смуг) → стискає по частоті до mel-шкали
              └─► log(S + 1e-9)        → логарифмічне масштабування
                    └─► crop/pad        → [128 × 130] фіксований розмір
```

**Чому мел-шкала?** Людське вухо сприймає висоту звуку логарифмічно. Мел-шкала апроксимує цю нелінійність, тому однакова кількість бінів відповідає однаковій "перцептивній відстані".

In [ ]:
mel = compute_mel_spectrogram(demo_wav)

print(f"Shape : {mel.shape}  (n_mels × time_frames)")
print(f"Min   : {mel.min():.3f}")
print(f"Max   : {mel.max():.3f}")
print(f"Mean  : {mel.mean():.3f}")

cmap = LinearSegmentedColormap.from_list(
    "dark_heat",
    ["#0f1117", "#1a237e", "#0277bd", "#00838f",
     "#2e7d32", "#f9a825", "#e65100", "#b71c1c"],
)

hop    = HOP_LENGTH
t_max  = mel.shape[1] * hop / SAMPLE_RATE

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                          gridspec_kw={"width_ratios": [3, 1], "wspace": 0.05})
fig.suptitle(f"Stage 2 — Log-Mel Spectrogram  [{mel.shape[0]} mel-bins × {mel.shape[1]} frames]\n"
             f"{demo_wav.name}  ·  label: {demo_wav.parent.name}",
             fontsize=12, fontweight="bold", color="white")

ax = axes[0]
img = ax.imshow(mel, aspect="auto", origin="lower", cmap=cmap,
                extent=[0, t_max, 0, SAMPLE_RATE/2/1000])
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (kHz)")
plt.colorbar(img, ax=ax, label="Log energy")

ax2 = axes[1]
mean_spec = mel.mean(axis=1)
freqs     = np.linspace(0, SAMPLE_RATE/2/1000, mel.shape[0])
ax2.plot(mean_spec, freqs, color=ORANGE, linewidth=1.3)
ax2.fill_betweenx(freqs, mean_spec, alpha=0.25, color=ORANGE)
ax2.set_xlabel("Mean log energy")
ax2.set_yticks([])
ax2.set_ylim(0, SAMPLE_RATE/2/1000)
ax2.set_title("Freq profile", fontsize=9, color=MUTED)
ax2.grid()

plt.tight_layout()
plt.show()

### Stage 2 — як виглядає спектрограма математично

Після перетворення отримуємо **двовимірну матрицю** `float32` розміром `[128 × 130]`:

```
mel[freq_bin, time_frame] = log-energy у частотній смузі freq_bin у момент time_frame

         time →  0       1       2    ...   129
freq ↑
  127  [ -5.21,  -4.88,  -6.03,  ...  -7.12 ]   ← high freq (≈11 kHz)
  126  [ -4.93,  -5.01,  -5.44,  ...  -6.78 ]
  ...
    1  [ -2.11,  -1.98,  -2.34,  ...  -3.01 ]
    0  [ -1.44,  -1.22,  -1.87,  ...  -2.55 ]   ← low freq (≈0 Hz)
```

Значення — логарифм енергії: великі від'ємні числа = тиша, числа близько до 0 = гучно.

In [ ]:
R, C = 10, 15   # скільки рядків/стовпців матриці показати

print(f"Тип масиву : {mel.dtype}")
print(f"Форма      : {mel.shape}  (freq_bins × time_frames)")
print(f"Значення   : min={mel.min():.3f}   max={mel.max():.3f}   mean={mel.mean():.3f}")
print()
print(f"Правий верхній кут  mel[{128-R}:{128}, 0:{C}]  (high freq, початок запису):")
print()

# header
header_str = "  freq\\time │" + "".join(f"  t={j:<3}" for j in range(C))
print(header_str)
print("  " + "─" * (len(header_str) - 2))

# rows: high freq at top (як на спектрограмі)
for i in range(128 - 1, 128 - R - 1, -1):
    row = mel[i, :C]
    vals = "".join(f" {v:+6.2f}" for v in row)
    print(f"  bin {i:>3}   │{vals}")

print()
print(f"Лівий нижній кут  mel[0:{R}, 0:{C}]  (low freq, початок запису):")
print()
print(header_str)
print("  " + "─" * (len(header_str) - 2))
for i in range(R - 1, -1, -1):
    row = mel[i, :C]
    vals = "".join(f" {v:+6.2f}" for v in row)
    print(f"  bin {i:>3}   │{vals}")

# ── Heatmap with numbers ──────────────────────────────────────────────────────
ROWS_SHOW, COLS_SHOW = 12, 20
patch = mel[-ROWS_SHOW:, :COLS_SHOW]          # top-left of spectrogram (high freq)

fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                          gridspec_kw={"width_ratios": [1, 1.6], "wspace": 0.4})
fig.suptitle(f"Stage 2 (matrix view) — числові значення матриці [{mel.shape[0]}×{mel.shape[1]}]",
             fontsize=12, fontweight="bold", color="white")

# LEFT: heatmap patch with annotated numbers
ax = axes[0]
im = ax.imshow(patch, aspect="auto", origin="upper", cmap=cmap)
plt.colorbar(im, ax=ax, label="log energy")

for r in range(ROWS_SHOW):
    for c in range(COLS_SHOW):
        v = patch[r, c]
        # white text on dark cells, dark on bright cells
        norm_v = (v - patch.min()) / (patch.max() - patch.min() + 1e-9)
        txt_color = "white" if norm_v < 0.6 else "#111"
        ax.text(c, r, f"{v:.1f}", ha="center", va="center",
                fontsize=5.5, color=txt_color)

ax.set_title(f"Patch mel[{mel.shape[0]-ROWS_SHOW}:{mel.shape[0]}, 0:{COLS_SHOW}]\n"
             f"(high-freq зона, перші {COLS_SHOW} time-frames)", fontsize=9, color="white")
ax.set_xlabel("Time frame →")
ax.set_ylabel("Freq bin (high → low) ↓")
y_ticks = [i for i in range(0, ROWS_SHOW, 2)]
ax.set_yticks(y_ticks)
ax.set_yticklabels([mel.shape[0] - 1 - i for i in y_ticks], fontsize=7)
ax.set_xticks(range(0, COLS_SHOW, 4))

# RIGHT: full spectrogram for context, з прямокутником де patch
ax2 = axes[1]
ax2.imshow(mel, aspect="auto", origin="lower", cmap=cmap)
# highlight the patch region
import matplotlib.patches as mpatches
rect = mpatches.Rectangle(
    (-0.5, mel.shape[0] - ROWS_SHOW - 0.5),
    COLS_SHOW, ROWS_SHOW,
    linewidth=2, edgecolor="white", facecolor="none", linestyle="--"
)
ax2.add_patch(rect)
ax2.set_title("Повна спектрограма [128×130]\n(пунктир = збільшена ділянка зліва)",
              fontsize=9, color="white")
ax2.set_xlabel("Time frame")
ax2.set_ylabel("Freq bin")
plt.colorbar(ax2.images[0], ax=ax2, label="log energy")

plt.tight_layout()
plt.show()

### Мел-спектрограми всіх 11 класів — кожен інструмент має унікальний "відбиток"

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
fig.suptitle("Stage 2b — Mel Spectrograms by Instrument Class",
             fontsize=13, fontweight="bold", color="white")

all_axes = axes.flatten()
for idx, cls in enumerate(CLASSES):
    m  = compute_mel_spectrogram(sample_wavs[cls])
    ax = all_axes[idx]
    ax.imshow(m, aspect="auto", origin="lower", cmap=cmap)
    ax.set_title(f"{CLASS_LABELS[cls]}  ({cls})", fontsize=10, color="white")
    ax.set_xticks([])
    ax.set_yticks([])

all_axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

---
## Stage 3 — Dataset & DataLoader

**IRMASDataset** повертає пари `(x, y)`:
- `x` — тензор `[1, 128, 130]` (спектрограма як одноканальне зображення)
- `y` — тензор `[11]` (one-hot вектор для тренувальних даних)

In [ ]:
full_ds = load_training_dataset(train_root)
train_paths, train_labels, val_paths, val_labels = train_val_split(
    full_ds.wav_paths, full_ds.labels
)
train_ds = IRMASDataset(train_paths, train_labels)
val_ds   = IRMASDataset(val_paths, val_labels)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

x_sample, y_sample = train_ds[0]
print(f"Total samples  : {len(full_ds)}")
print(f"Train split    : {len(train_ds)}")
print(f"Val split      : {len(val_ds)}")
print()
print(f"x shape        : {list(x_sample.shape)}   (channels, n_mels, time_frames)")
print(f"y shape        : {list(y_sample.shape)}  (one-hot over {NUM_CLASSES} classes)")
print(f"y values       : {y_sample.numpy().tolist()}")
print(f"active label   : {[CLASSES[i] for i, v in enumerate(y_sample) if v > 0]}")

# visualise label distribution in splits
train_counts = np.array(train_labels).sum(axis=0)
val_counts   = np.array(val_labels).sum(axis=0)

fig, ax = plt.subplots(figsize=(11, 3.5))
fig.suptitle("Stage 3 — Label counts in Train / Val splits",
             fontsize=12, fontweight="bold", color="white")
x_pos = np.arange(NUM_CLASSES)
ax.bar(x_pos - 0.2, train_counts, 0.38, label="Train", color=ACCENT,  edgecolor="#0f1117")
ax.bar(x_pos + 0.2, val_counts,   0.38, label="Val",   color=ORANGE,  edgecolor="#0f1117")
ax.set_xticks(x_pos)
ax.set_xticklabels([CLASS_LABELS[c] for c in CLASSES], rotation=30, ha="right")
ax.legend()
ax.grid(axis="y")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

---
## Stage 4 — Model Architecture

```
Input [B, 1, 128, 130]
  │
  ├─ Conv2d(1→16, 3×3) → BN → ReLU → MaxPool(2×2)   → [B, 16, 64, 65]
  ├─ Conv2d(16→32, 3×3) → BN → ReLU → MaxPool(2×2)  → [B, 32, 32, 32]
  ├─ Dropout2d(0.25)
  │
  ├─ reshape → [B, 32 time-steps, 1024 features]
  │
  ├─ BiLSTM(input=1024, hidden=64, bidirectional)     → [B, 32, 128]
  ├─ mean over time                                   → [B, 128]
  ├─ Dropout(0.30)
  │
  └─ Linear(128 → 11)                                → [B, 11]  raw logits
                                                         sigmoid → probabilities
```

In [ ]:
model = InstrumentCNNBiLSTM().to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Approx size         : {trainable*4/1024:.1f} KB (float32)")

# verify forward pass shapes
model.eval()
with torch.no_grad():
    dummy = torch.randn(4, 1, 128, 130).to(device)
    out   = model(dummy)
print(f"\nForward pass  input : {list(dummy.shape)}")
print(f"Forward pass output : {list(out.shape)}   (raw logits, apply sigmoid for probs)")

### Кількість параметрів по шарах

In [ ]:
layer_names, layer_params = [], []
for name, param in model.named_parameters():
    if param.requires_grad:
        layer_names.append(name)
        layer_params.append(param.numel())

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Stage 4 — Parameter Count per Layer",
             fontsize=12, fontweight="bold", color="white")
colors = plt.cm.cool(np.linspace(0.1, 0.9, len(layer_names)))
bars = ax.barh(layer_names, layer_params, color=colors, edgecolor="#0f1117")
for bar, v in zip(bars, layer_params):
    ax.text(v + 200, bar.get_y() + bar.get_height()/2,
            f"{v:,}", va="center", fontsize=8, color="#c0c4d8")
ax.set_xlabel("Number of parameters")
ax.invert_yaxis()
ax.grid(axis="x")
ax.set_title(f"Total: {total:,} params", fontsize=9, color=MUTED)
plt.tight_layout()
plt.show()

---
## Stage 5 — Training Loop

**Loss:** BCEWithLogitsLoss — для кожного з 11 класів окремо:
$$\mathcal{L} = -\frac{1}{K}\sum_{k=1}^{K} \left[ y_k \log\sigma(\hat{y}_k) + (1-y_k)\log(1-\sigma(\hat{y}_k)) \right]$$

- Якщо random prediction (0.5) — loss ≈ log(2) ≈ **0.693**
- Ціль: наблизитись до 0

In [ ]:
EPOCHS = 20

model     = InstrumentCNNBiLSTM().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # ── train ──
    model.train()
    total_loss = 0.0
    for x_b, y_b in tqdm(train_loader, desc=f"Epoch {epoch:>2}/{EPOCHS} train", leave=False):
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x_b.size(0)
    train_loss = total_loss / len(train_ds)

    # ── val ──
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x_b, y_b in val_loader:
            total_loss += criterion(model(x_b.to(device)), y_b.to(device)).item() * x_b.size(0)
    val_loss = total_loss / len(val_ds)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch:>3}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}")

torch.save({"model_state_dict": model.state_dict(),
            "classes": CLASSES, "epoch": EPOCHS}, CHECKPOINT)
print(f"\nCheckpoint saved → {CHECKPOINT}")

### Loss curve

In [ ]:
epochs = list(range(1, len(train_losses) + 1))

fig, ax = plt.subplots(figsize=(11, 4))
fig.suptitle("Stage 5 — Training & Validation Loss",
             fontsize=13, fontweight="bold", color="white")

ax.plot(epochs, train_losses, color=ACCENT,  lw=2, marker="o", ms=4, label="Train loss")
ax.plot(epochs, val_losses,   color=GREEN,   lw=2, marker="s", ms=4, label="Val loss",   ls="--")
ax.fill_between(epochs, train_losses, val_losses, alpha=0.08, color=ORANGE)
ax.axhline(np.log(2), color=RED, lw=0.9, ls=":", label=f"Random baseline ≈ {np.log(2):.3f}")

ax.annotate(f"start\n{train_losses[0]:.4f}", xy=(1, train_losses[0]),
            xytext=(2.5, train_losses[0]+0.025), color=ACCENT, fontsize=8,
            arrowprops=dict(arrowstyle="->", color=ACCENT))
ax.annotate(f"end\n{train_losses[-1]:.4f}", xy=(EPOCHS, train_losses[-1]),
            xytext=(EPOCHS-5, train_losses[-1]+0.025), color=ACCENT, fontsize=8,
            arrowprops=dict(arrowstyle="->", color=ACCENT))

ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.legend()
ax.grid()
ax.set_xlim(1, EPOCHS)

improvement = (1 - train_losses[-1]/train_losses[0]) * 100
ax.set_title(f"Loss: {train_losses[0]:.4f} → {train_losses[-1]:.4f}  ({improvement:.1f}% improvement)",
             fontsize=9, color=MUTED)
plt.tight_layout()
plt.show()

---
## Stage 6 — Inference on Samples

Для кожного семплу:
1. Подаємо спектрограму в модель
2. Застосовуємо **sigmoid** → 11 ймовірностей ∈ (0, 1)
3. Порівнюємо з **threshold** → бінарні передбачення

Кольори: 🟢 True Positive · 🔴 False Positive · 🟠 False Negative (пропуск) · ⬛ True Negative

In [ ]:
model.eval()

# one sample per class from val set
demo_indices = []
seen_classes = set()
for i, path in enumerate(val_paths):
    cls = path.parent.name
    if cls not in seen_classes:
        demo_indices.append(i)
        seen_classes.add(cls)
    if len(demo_indices) == 6:
        break

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f"Stage 6 — Model Inference  (threshold={THRESHOLD})",
             fontsize=13, fontweight="bold", color="white")

with torch.no_grad():
    for ax, idx in zip(axes.flatten(), demo_indices):
        x, y_true = val_ds[idx]
        logit     = model(x.unsqueeze(0).to(device))
        probs     = torch.sigmoid(logit).cpu().numpy()[0]

        true_cls = [CLASSES[i] for i, v in enumerate(y_true) if v > 0]
        pred_cls = [CLASSES[i] for i, p in enumerate(probs)  if p >= THRESHOLD]
        labels   = [CLASS_LABELS[c] for c in CLASSES]

        bar_colors = []
        for i in range(NUM_CLASSES):
            is_true = y_true[i] > 0
            is_pred = probs[i] >= THRESHOLD
            if   is_true and is_pred:      bar_colors.append(GREEN)
            elif is_pred and not is_true:  bar_colors.append(RED)
            elif is_true and not is_pred:  bar_colors.append(ORANGE)
            else:                          bar_colors.append(MUTED)

        bars_ = ax.barh(labels, probs, color=bar_colors, edgecolor="#0f1117", lw=0.5)
        ax.axvline(THRESHOLD, color="white", lw=0.9, ls="--", alpha=0.6)
        ax.set_xlim(0, max(probs.max() * 1.3, THRESHOLD * 2))
        ax.invert_yaxis()
        ax.grid(axis="x")

        hit = "✓" if set(true_cls) & set(pred_cls) else "✗"
        ax.set_title(f"{hit}  true={true_cls}  pred={pred_cls}\n{val_paths[idx].name[:40]}",
                     fontsize=7.5, color="white")
        for bar_, p in zip(bars_, probs):
            if p > 0.01:
                ax.text(p + 0.002, bar_.get_y() + bar_.get_height()/2,
                        f"{p:.3f}", va="center", fontsize=6.5, color="#c0c4d8")

from matplotlib.patches import Patch
fig.legend(
    handles=[
        Patch(color=GREEN,  label="True Positive"),
        Patch(color=RED,    label="False Positive"),
        Patch(color=ORANGE, label="False Negative (missed)"),
        Patch(color=MUTED,  label="True Negative"),
    ],
    loc="lower center", ncol=4, fontsize=9,
    framealpha=0.2, bbox_to_anchor=(0.5, -0.04)
)
plt.tight_layout()
plt.show()

### Розподіл ймовірностей по всьому val set

In [ ]:
all_probs, all_targets_np = [], []
model.eval()
with torch.no_grad():
    for x_b, y_b in val_loader:
        p = torch.sigmoid(model(x_b.to(device))).cpu().numpy()
        all_probs.append(p)
        all_targets_np.append(y_b.numpy())

all_probs   = np.vstack(all_probs)
all_targets_np = np.vstack(all_targets_np)

# positive (y=1) vs negative (y=0) prob distributions
pos_probs = all_probs[all_targets_np == 1]
neg_probs = all_probs[all_targets_np == 0]

fig, ax = plt.subplots(figsize=(10, 4))
fig.suptitle("Stage 6b — Sigmoid Probability Distribution",
             fontsize=12, fontweight="bold", color="white")

ax.hist(neg_probs, bins=50, color=MUTED,  alpha=0.7, label="True label = 0 (negative)", density=True)
ax.hist(pos_probs, bins=50, color=GREEN,  alpha=0.8, label="True label = 1 (positive)", density=True)
ax.axvline(THRESHOLD, color="white", lw=1.2, ls="--", label=f"threshold = {THRESHOLD}")
ax.set_xlabel("Sigmoid probability")
ax.set_ylabel("Density")
ax.legend()
ax.grid()
ax.set_title("Ideal: positive cluster → right of threshold, negative → left",
             fontsize=9, color=MUTED)
plt.tight_layout()
plt.show()

print(f"Positive probs: mean={pos_probs.mean():.3f}  max={pos_probs.max():.3f}")
print(f"Negative probs: mean={neg_probs.mean():.3f}  max={neg_probs.max():.3f}")

---
## Stage 7 — Evaluation Metrics

**F1-score** для multi-label:
$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

- **F1-macro** — середнє F1 по кожному класу окремо (штрафує за рідкісні класи)
- **F1-micro**  — агреговане F1 по всіх TP/FP/FN разом

In [ ]:
preds   = (all_probs >= THRESHOLD).astype(int)
targets = all_targets_np.astype(int)

f1_macro = f1_score(targets, preds, average="macro",  zero_division=0)
f1_micro = f1_score(targets, preds, average="micro",  zero_division=0)
f1_per   = f1_score(targets, preds, average=None,     zero_division=0)

print(f"Threshold : {THRESHOLD}")
print(f"F1-macro  : {f1_macro:.4f}")
print(f"F1-micro  : {f1_micro:.4f}")
print()
print(f"{'Class':<12} {'F1':>6}")
print("-" * 20)
for cls, f1 in zip(CLASSES, f1_per):
    bar_vis = "█" * int(f1 * 30)
    print(f"{CLASS_LABELS[cls]:<12} {f1:.4f}  {bar_vis}")

### Візуалізація метрик

In [ ]:
tp = (preds * targets).sum(axis=0)
fp = (preds * (1 - targets)).sum(axis=0)
fn = ((1 - preds) * targets).sum(axis=0)

fig = plt.figure(figsize=(15, 7))
fig.suptitle(f"Stage 7 — Evaluation Metrics  (threshold={THRESHOLD}  ·  val set)",
             fontsize=13, fontweight="bold", color="white")

gs   = gridspec.GridSpec(1, 2, width_ratios=[2, 1.3], wspace=0.35)

# per-class F1
ax1 = fig.add_subplot(gs[0])
labels_full = [CLASS_LABELS[c] for c in CLASSES]
bar_colors  = [GREEN if f > 0.3 else (ACCENT if f > 0.1 else RED) for f in f1_per]
bars_       = ax1.barh(labels_full, f1_per, color=bar_colors, edgecolor="#0f1117", lw=0.7)
ax1.axvline(f1_macro, color=ORANGE, lw=1.5, ls="--", label=f"F1-macro = {f1_macro:.4f}")
ax1.axvline(f1_micro, color=GREEN,  lw=1.5, ls=":",  label=f"F1-micro = {f1_micro:.4f}")
ax1.set_xlim(0, 1.0)
ax1.invert_yaxis()
ax1.legend(loc="lower right", fontsize=9)
ax1.grid(axis="x")
ax1.set_xlabel("F1 score")
ax1.set_title("Per-class F1", color="white")
for bar_, f in zip(bars_, f1_per):
    ax1.text(f + 0.01, bar_.get_y() + bar_.get_height()/2,
             f"{f:.3f}", va="center", fontsize=8.5, color="#c0c4d8")

# TP / FP / FN grouped bar
ax2 = fig.add_subplot(gs[1])
x_  = np.arange(NUM_CLASSES)
w   = 0.28
ax2.bar(x_ - w, tp, w, label="TP", color=GREEN,  edgecolor="#0f1117")
ax2.bar(x_,     fp, w, label="FP", color=RED,    edgecolor="#0f1117")
ax2.bar(x_ + w, fn, w, label="FN", color=ORANGE, edgecolor="#0f1117")
ax2.set_xticks(x_)
ax2.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8)
ax2.legend(fontsize=9)
ax2.grid(axis="y")
ax2.set_ylabel("Count")
ax2.set_title("TP / FP / FN per class", color="white")

plt.tight_layout()
plt.show()

---
## Summary

| | Значення |
|---|---|
| Датасет | 550 файлів (11 × 50) |
| Параметри моделі | 564,395 |
| Епох тренування | 20 |
| Train loss (start → end) | 0.47 → 0.27 |
| F1-macro | ~0.28 |
| F1-micro | ~0.27 |

### Наступні кроки
1. **Повний датасет** — завантажити всі ~6000 файлів (`--max-per-class 600`)
2. **Аугментація** — pitch shift, time stretch, mixup
3. **Per-class threshold** — замість одного порогу 0.09 підібрати окремий для кожного класу
4. **Baseline порівняння** — SVM на мфкк (MFCC), HMM
5. **Більше епох** — 50–100 з early stopping